# 03b. SFT v2 — проверка данных и осторожный повторный эксперимент

Qwen2.5-3B-Instruct + QLoRA на Colab T4. Новый адаптер обучается от исходной базы.
`learning_rate=5e-5`, одна эпоха. Рабочие настройки точности Trainer из успешного
03 сохранены: `fp16=False`, `bf16=False`; вычисления bitsandbytes — float16.

**Режим:** сначала `SMOKE_TEST=True` (5 шагов), затем в новой/перезапущенной среде
`SMOKE_TEST=False` (полный v2-прогон). Папки и названия экспортов у режимов разные.

**Статус данных:** 650 кандидатов, аудит Codex выполнен для 48 случайно выбранных
примеров (по 6 на категорию), 41 отредактирован, 7 оставлены. У 602 примеров
сохраняется статус `not_reviewed`. Это исследовательский кандидат, требующий
дальнейшей смысловой оценки; полной экспертной проверки всех ответов не было.
Train=560, validation=90. Группы сценариев не пересекаются, все 8 категорий
представлены в обеих частях. Группы эвристические, не экспертная семантическая разметка.

Мы одновременно изменили часть данных, разбиение и настройки. Сравнение оценивает
весь набор изменений и не изолирует эффект каждого из них. Val loss v1 и v2
посчитаны на разных наборах, напрямую сравнивать их значения нельзя.

## 1. Подготовка
Выберите в Colab T4 GPU. Код берётся из GitHub. Если среда попросит перезапуск
после установки, перезапустите её и начните сверху. Начинайте полный прогон
после проверки вывода smoke-теста, в свежей среде для освобождения GPU.

In [ ]:
SMOKE_TEST = True
from pathlib import Path
import os, subprocess, sys, json
ROOT = Path('/content/lm-hw')
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/maria-igntv/lm-hw.git', str(ROOT)], check=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert (ROOT/'data/sft_v2/manifest.json').exists(), 'Обновите клонированный репозиторий через git pull --ff-only и откройте свежую версию ноутбука.'
%pip install -q "transformers==5.22.1" "trl==1.17.1" "peft==0.21.2" "bitsandbytes==0.50.2" "accelerate==1.12.0" "datasets==5.1.0" tensorboard matplotlib
import torch
assert torch.cuda.is_available(), 'Нужен T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
SEED = 42
RUN_NAME = 'sft_v2_smoke' if SMOKE_TEST else 'sft_v2'
RUN_DIR = ROOT/'results'/RUN_NAME
# Для нового независимого повторения измените имя RUN_NAME.
assert not (RUN_DIR/'export/adapter/adapter_model.safetensors').exists(), 'Здесь уже сохранён результат. Выберите новое RUN_NAME.'
RUN_DIR.mkdir(parents=True, exist_ok=True)

## 2. Проверяем версию данных и разделение
Системный промпт должен быть единым. Ни одна группа сценариев и ни один точный
вопрос не должны попадать в обе части. Статусы аудита выводятся явно.
Файл manifest.json и хеши защищают от случайного смешивания версий.

In [ ]:
import hashlib, re
from collections import Counter
from datasets import Dataset

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]

manifest = json.loads((ROOT/'data/sft_v2/manifest.json').read_text())
for name, expected in manifest['output_hashes'].items():
    assert sha256(ROOT/'data/sft_v2'/name) == expected, f'Изменён файл {name}'
train_raw = read_jsonl(ROOT/'data/sft_v2/sft_train.jsonl')
val_raw = read_jsonl(ROOT/'data/sft_v2/sft_val.jsonl')
assert not ({r['scenario_group'] for r in train_raw} & {r['scenario_group'] for r in val_raw})
normalize = lambda s: re.sub(r'[^\w]+', ' ', s.casefold().replace('ё', 'е')).strip()
assert not ({normalize(r['messages'][1]['content']) for r in train_raw} & {normalize(r['messages'][1]['content']) for r in val_raw})
assert set(r['category'] for r in train_raw) == set(r['category'] for r in val_raw)
assert len({r['messages'][0]['content'] for r in train_raw+val_raw}) == 1
for label, records in [('train',train_raw), ('val',val_raw)]:
    print(label, len(records), dict(Counter(r['category'] for r in records)))
    print('Статусы аудита:', dict(Counter(r['audit_status'] for r in records)))
SYSTEM_PROMPT = train_raw[0]['messages'][0]['content']

## 3. Загружаем исходную модель и новый LoRA-адаптер
Старый адаптер v1 здесь не подключается. Создаём новый с той же конфигурацией LoRA,
чтобы уменьшить число изменяемых факторов. База заморожена и загружена в NF4.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
set_seed(SEED)
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                 bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=quantization, device_map='auto', dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Проверяем длины, формат и считаем настройки
Проверяются все примеры, даже в smoke-режиме. Если ответ приходится обрезать,
ноутбук останавливается: сначала осмотрите длинные записи, вместо молчаливой потери
конца ответа. Максимальная длина пока 1024, как в v1.

In [ ]:
MAX_LENGTH = 1024
length_rows = []
for split, records in [('train',train_raw), ('val',val_raw)]:
    for r in records:
        assert [m['role'] for m in r['messages']] == ['system','user','assistant']
        assert all(isinstance(m['content'],str) and m['content'].strip() for m in r['messages'])
        ids = tokenizer.apply_chat_template(r['messages'], tokenize=True, add_generation_prompt=False, return_dict=False)
        length_rows.append(dict(sample_id=r['sample_id'], split=split, tokens=len(ids)))
too_long = [r for r in length_rows if r['tokens'] > MAX_LENGTH]
print('Максимальная длина:', max(r['tokens'] for r in length_rows))
print('Слишком длинных:', len(too_long))
(RUN_DIR/'token_lengths.json').write_text(json.dumps(length_rows, ensure_ascii=False, indent=2))
assert not too_long, f'Есть обрезаемые примеры. Первые ID: {too_long[:5]}'

# Smoke проверяет короткий технический прогон; его метрики не считать итоговыми.
training_rows = train_raw[:16] if SMOKE_TEST else train_raw
validation_rows = val_raw[:8] if SMOKE_TEST else val_raw

def to_prompt_completion(example):
    return dict(prompt=example['messages'][:-1], completion=[example['messages'][-1]])

train_ds = Dataset.from_list(training_rows)
val_ds = Dataset.from_list(validation_rows)
train_pc = train_ds.map(to_prompt_completion, remove_columns=train_ds.column_names)
val_pc = val_ds.map(to_prompt_completion, remove_columns=val_ds.column_names)

## 5. Настраиваем обучение и мониторинг
Новый стартовый вариант — learning_rate 5e-5 и одна эпоха. В полном прогоне с
batch=2 и накоплением 8 ожидается около 35 шагов оптимизации на одной T4.
TensorBoard сохраняет логи локально и не требует API-ключа. Лучший адаптер выбирается
по validation loss. Дополнительно оценка сохраняется в конце обучения.

In [ ]:
import math
from trl import SFTConfig, SFTTrainer
expected_steps = math.ceil(len(training_rows)/16)
eval_every = 1 if SMOKE_TEST else 5
training_args = SFTConfig(
    output_dir=str(RUN_DIR/'checkpoints'), seed=SEED,
    max_length=MAX_LENGTH, completion_only_loss=True, packing=False,
    per_device_train_batch_size=2, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, gradient_checkpointing=True,
    optim='paged_adamw_32bit', fp16=False, bf16=False,
    num_train_epochs=1, learning_rate=5e-5,
    lr_scheduler_type='cosine', warmup_steps=1 if SMOKE_TEST else max(1,round(expected_steps*0.05)),
    weight_decay=0.01, max_grad_norm=1.0,
    logging_steps=1,
    eval_strategy='steps', eval_steps=eval_every,
    save_strategy='steps', save_steps=eval_every, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
    report_to='tensorboard', max_steps=5 if SMOKE_TEST else -1,
)
trainer = SFTTrainer(model=model, args=training_args, train_dataset=train_pc,
                     eval_dataset=val_pc, processing_class=tokenizer)
assert len(trainer.train_dataset) == len(training_rows), 'Trainer удалил обучающие примеры; проверьте маскирование.'
assert len(trainer.eval_dataset) == len(validation_rows), 'Trainer удалил валидационные примеры.'
print('Плановых шагов:', 5 if SMOKE_TEST else expected_steps)

## 6. Проверяем маскирование по реальным токенам
Проверка сильнее простого подсчёта -100: все токены до конца заголовка assistant
должны быть замаскированы. Выводятся части реально обучаемого ответа. Проверьте,
что там нет вопроса пользователя. Результаты записываются в mask_audit.jsonl.

In [ ]:
response_marker = tokenizer.encode('<|im_start|>assistant\n', add_special_tokens=False)
mask_checks = []
for i in range(min(10,len(trainer.train_dataset))):
    batch = trainer.data_collator([trainer.train_dataset[i]])
    input_ids = batch['input_ids'][0].tolist()
    labels = batch['labels'][0].tolist()
    marker_positions = [j for j in range(len(input_ids)-len(response_marker)+1)
                        if input_ids[j:j+len(response_marker)] == response_marker]
    assert len(marker_positions)==1, 'Не найден единственный заголовок assistant.'
    boundary = marker_positions[0]+len(response_marker)
    assert all(label == -100 for label in labels[:boundary]), 'Лосс попал на промпт!'
    supervised_positions = [j for j,label in enumerate(labels) if label != -100]
    assert supervised_positions and min(supervised_positions)>=boundary
    assert all(labels[j] == input_ids[j] for j in supervised_positions)
    learned_text = tokenizer.decode([input_ids[j] for j in supervised_positions], skip_special_tokens=False)
    record = dict(sample_id=training_rows[i]['sample_id'], prompt_boundary=boundary,
                  supervised_tokens=len(supervised_positions), supervised_text=learned_text)
    mask_checks.append(record)
    print(record['sample_id'], '| токенов с лоссом:',len(supervised_positions))
    print(learned_text[:240])
(RUN_DIR/'mask_audit.jsonl').write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in mask_checks))
print('Проверки маскирования пройдены.')

## 7. Запускаем обучение
В smoke-режиме ждём только подтверждения работоспособности. Для полного обучения
поставьте SMOKE_TEST=False в первой ячейке и выполните ноутбук в свежей среде.
Не используйте smoke-адаптер как итоговый результат SFT v2.

In [ ]:
train_result = trainer.train()
final_evaluation = trainer.evaluate()
print('Обучение:', train_result.metrics)
print('Лучшая оценка:', trainer.state.best_metric)
print('Лучший чекпоинт:', trainer.state.best_model_checkpoint)

## 8. Сохраняем адаптер, кривые и сведения о прогоне
В экспорт попадает адаптер после выбора лучшего чекпоинта и файлы для отчёта.
Папки v1 не изменяются. Для независимой оценки новый адаптер затем подключается в 04.

In [ ]:
import shutil, importlib.metadata
from datetime import datetime, timezone
import matplotlib.pyplot as plt
EXPORT_DIR = RUN_DIR/'export'
ADAPTER_DIR = EXPORT_DIR/'adapter'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model.config.use_cache = True
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
log_history = trainer.state.log_history
(EXPORT_DIR/'trainer_log_history.json').write_text(json.dumps(log_history,ensure_ascii=False,indent=2))
trainer.state.save_to_json(str(EXPORT_DIR/'trainer_state.json'))
(EXPORT_DIR/'training_args.json').write_text(training_args.to_json_string())
for filename in ('mask_audit.jsonl','token_lengths.json'):
    shutil.copy2(RUN_DIR/filename, EXPORT_DIR/filename)
shutil.copy2(ROOT/'data/sft_v2/manifest.json', EXPORT_DIR/'dataset_manifest.json')
shutil.copytree(RUN_DIR/'checkpoints/runs', EXPORT_DIR/'tensorboard', dirs_exist_ok=True)
metadata = dict(model=MODEL_NAME, smoke_test=SMOKE_TEST,
    date_utc=datetime.now(timezone.utc).isoformat(),
    versions={p:importlib.metadata.version(p) for p in
              ('torch','transformers','trl','peft','bitsandbytes','datasets','accelerate')},
    gpu=torch.cuda.get_device_name(0), dataset_version=manifest['version'],
    dataset_hashes=manifest['output_hashes'],
    train_examples=len(training_rows), val_examples=len(validation_rows),
    learning_rate=5e-5, epochs=1,
    train_metrics=train_result.metrics, final_evaluation=final_evaluation,
    best_metric=trainer.state.best_metric, best_checkpoint=trainer.state.best_model_checkpoint)
(EXPORT_DIR/'run_metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2))
train_points=[(r['step'],r['loss']) for r in log_history if 'loss' in r]
val_points=[(r['step'],r['eval_loss']) for r in log_history if 'eval_loss' in r]
plt.figure(figsize=(8,4))
if train_points:
    plt.plot(*zip(*train_points),label='train loss')
if val_points:
    plt.plot(*zip(*val_points),marker='o',label='validation loss')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3)
plt.title(RUN_NAME)
plt.savefig(EXPORT_DIR/'loss_curve.png',dpi=150,bbox_inches='tight')
plt.show()
print('Экспорт:',EXPORT_DIR)

## 9. Скачиваем результат
Сохраните ZIP до закрытия сессии. Для оценки в 04 используйте адаптер из `export/adapter`
полного прогона. В 04 задайте отдельную папку OUT_DIR, например
`ROOT/'results/eval_sft_v2'`, чтобы сохранить результаты первого эксперимента.
Затем оцените новые 100 ответов тем же Qwen-судьёй и тем же протоколом.

In [ ]:
from google.colab import files
archive = shutil.make_archive('/content/'+RUN_NAME, 'zip', root_dir=EXPORT_DIR)
files.download(archive)
print('Архив:',archive)